# 03 · Join Sofascore + Capology — England Premier League 20/21

Integración de estadísticas de rendimiento (Sofascore) con datos salariales (Capology)
para la temporada **2020/21 de Premier League inglesa**.

**Flujo de matching:**
1. Normalización de nombres (tildes, mayúsculas, caracteres especiales)
2. `TEAM_MAP`: alineación manual de nombres de equipo entre fuentes
3. Merge exacto normalizado
4. Fuzzy matching en cuatro niveles:
   - Score ≥ 0.90 → aceptación automática
   - 0.75 ≤ score < 0.90 → revisión manual
   - 0.50 ≤ score < 0.75 → revisión manual estricta
   - score < 0.50 → revisión manual muy estricta
5. Revisión de jugadores sin salario
6. Guardado en `data/master/`

---

## 1. Imports y rutas

In [1]:
import pandas as pd
import unicodedata
import re
from rapidfuzz import fuzz
from pathlib import Path
from IPython.display import display

ROOT       = Path.cwd().parents[1]
SF_DIR     = ROOT / 'data' / 'processed' / 'sofascore'
CG_DIR     = ROOT / 'data' / 'processed' / 'capology'
MASTER_DIR = ROOT / 'data' / 'master'
MASTER_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Rutas configuradas')
print(f'   Root:   {ROOT}')
print(f'   Master: {MASTER_DIR}')

✅ Rutas configuradas
   Root:   d:\USER\Desktop\TFM
   Master: d:\USER\Desktop\TFM\data\master


## 2. Función de normalización

In [2]:
def normalize(s):
    """
    Normaliza un string para comparación: elimina tildes, pasa a minúsculas,
    elimina caracteres especiales y espacios extra.
    """
    if pd.isna(s):
        return ''
    s = str(s)
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    s = re.sub(r'[^a-z0-9\s]', ' ', s.lower().strip())
    return re.sub(r'\s+', ' ', s).strip()

print('✅ Función definida')

✅ Función definida


## 3. Carga de datos

In [3]:
df_sf = pd.read_csv(SF_DIR / 'df_england_2021.csv').copy()
df_cg = pd.read_csv(CG_DIR / 'cg_england_2021.csv').copy()

print(f'Sofascore:  {df_sf.shape[0]} jugadores | {df_sf.shape[1]} columnas')
print(f'Capology:   {df_cg.shape[0]} jugadores | {df_cg.shape[1]} columnas')

Sofascore:  524 jugadores | 116 columnas
Capology:   593 jugadores | 9 columnas


## 4. Normalización

In [4]:
df_sf['player_norm'] = df_sf['player'].apply(normalize)
df_sf['team_norm']   = df_sf['team'].apply(normalize)
df_cg['player_norm'] = df_cg['player'].apply(normalize)
df_cg['team_norm']   = df_cg['club'].apply(normalize)

print('✅ Normalización aplicada')

✅ Normalización aplicada


## 5. Alineación de equipos (TEAM_MAP)

### 5.1 Identificar discrepancias de nombres de equipo

In [5]:
solo_sf = set(df_sf['team_norm'].unique()) - set(df_cg['team_norm'].unique())
solo_cg = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())

print('En Sofascore pero no en Capology:')
for e in sorted(solo_sf): print(f'   {e}')
print()
print('En Capology pero no en Sofascore:')
for e in sorted(solo_cg): print(f'   {e}')

En Sofascore pero no en Capology:
   brighton hove albion
   leeds united
   leicester city
   newcastle united
   tottenham hotspur
   west bromwich albion
   west ham united

En Capology pero no en Sofascore:
   brighton
   leeds
   leicester
   newcastle
   tottenham
   west bromwich
   west ham


### 5.2 Aplicar TEAM_MAP

Rellenar con las discrepancias identificadas en la celda anterior.

In [6]:
# ── Ajustar según la celda anterior ──────────────────────────
TEAM_MAP = {'brighton':'brighton hove albion',
            'leeds':'leeds united',
            'leicester':'leicester city',
            'newcastle':'newcastle united',
            'tottenham':'tottenham hotspur',
            'west bromwich':'west bromwich albion',
            'west ham':'west ham united'

}
# ─────────────────────────────────────────────────────────────

df_cg['team_norm'] = df_cg['team_norm'].replace(TEAM_MAP)

diff = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())
if diff:
    print(f'⚠️  Equipos de CG aún sin match en SF: {diff}')
else:
    print('✅ Todos los equipos alineados')


✅ Todos los equipos alineados


## 6. Merge exacto normalizado

In [7]:
df_merged = df_sf.merge(
    df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
            'position', 'age', 'nationality']],
    on=['player_norm', 'team_norm'],
    how='left'
)

matched = df_merged['gross_annual_eur'].notna().sum()
total   = len(df_merged)

print(f'Merge exacto: {matched}/{total} ({matched/total:.1%})')
print(f'Sin emparejar: {total - matched}')

Merge exacto: 478/525 (91.0%)
Sin emparejar: 47


## 7. Fuzzy matching sobre los no emparejados

Se generan candidatos para todos los jugadores sin match exacto,
sin umbral mínimo, y se clasifican en cuatro niveles.

In [8]:
df_unmatched = df_merged[df_merged['gross_annual_eur'].isna()].copy()
cg_by_team   = df_cg.groupby('team_norm')['player_norm'].apply(list).to_dict()

rows = []
for _, row in df_unmatched[['player','team','player_norm','team_norm']].drop_duplicates().iterrows():
    candidates = cg_by_team.get(row['team_norm'], [])
    best_match, best_score = None, 0
    for cand in candidates:
        score = fuzz.ratio(row['player_norm'], cand) / 100
        if score > best_score:
            best_score = score
            best_match = cand
    if best_match is not None:
        rows.append({
            'player_sf'  : row['player'],
            'team'       : row['team'],
            'player_norm': row['player_norm'],
            'team_norm'  : row['team_norm'],
            'cg_match'   : best_match,
            'score'      : round(best_score, 3)
        })

df_candidates = pd.DataFrame(rows).sort_values('score', ascending=False)
auto_matches     = df_candidates[df_candidates['score'] >= 0.90].copy()
review_matches   = df_candidates[(df_candidates['score'] >= 0.75) & (df_candidates['score'] < 0.90)].copy()
low_matches      = df_candidates[(df_candidates['score'] >= 0.50) & (df_candidates['score'] < 0.75)].copy()
very_low_matches = df_candidates[df_candidates['score'] < 0.50].copy()

print(f'Auto-aceptados    (score ≥ 0.90):          {len(auto_matches)}')
print(f'Revisión media    (0.75 ≤ score < 0.90):   {len(review_matches)}')
print(f'Revisión estricta (0.50 ≤ score < 0.75):   {len(low_matches)}')
print(f'Revisión muy est. (score < 0.50):           {len(very_low_matches)}')

Auto-aceptados    (score ≥ 0.90):          5
Revisión media    (0.75 ≤ score < 0.90):   3
Revisión estricta (0.50 ≤ score < 0.75):   22
Revisión muy est. (score < 0.50):           17


### 7.1 Matches automáticos (score ≥ 0.90)

Revisar para confirmar que todos son correctos.

In [9]:
auto_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
1,Pierre-Emile Højbjerg,Tottenham Hotspur,pierre emile hojbjerg,0.976
5,Gylfi Sigurðsson,Everton,gylfi sigurdsson,0.968
7,Łukasz Fabiański,West Ham United,lukasz fabianski,0.968
17,Andriy Yarmolenko,West Ham United,andrii yarmolenko,0.941
26,Joshua Onomah,Fulham,josh onomah,0.917


### 7.2 Revisión media (0.75 ≤ score < 0.90)

Añadir a `EXCLUDE_FROM_FUZZY` el `player_norm` de los incorrectos.

In [10]:
review_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
11,Jóhann Guðmundsson,Burnley,johann berg gudmundsson,0.850
8,Edward Nketiah,Arsenal,eddie nketiah,0.815
12,Ian Carlo Poveda,Leeds United,ian poveda,0.769


In [11]:
# ── Falsos positivos a excluir del nivel medio ────────────────
EXCLUDE_FROM_FUZZY = [

]
# ─────────────────────────────────────────────────────────────

review_accepted = review_matches[~review_matches['player_norm'].isin(EXCLUDE_FROM_FUZZY)]
print(f'Aceptados: {len(review_accepted)} | Excluidos: {len(EXCLUDE_FROM_FUZZY)}')


Aceptados: 3 | Excluidos: 0


### 7.3 Revisión estricta (0.50 ≤ score < 0.75)

Por defecto ninguno se acepta. Añadir a `ACCEPT_LOW_FUZZY` los correctos.

In [12]:
low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
33,Moise Kean,Everton,michael keane,0.696
9,Mahmoud Trézéguet,Aston Villa,trezeguet,0.692
2,Anthony Elanga,Manchester United,anthony martial,0.690
32,Rúben Vinagre,Wolverhampton,ruben neves,0.667
10,Bobby Decordova-Reid,Fulham,bobby reid,0.667
4,Frank Anguissa,Fulham,andre zambo anguissa,0.647
35,Josh Benson,Burnley,josh brownhill,0.640
31,Wesley Moraes,Aston Villa,wesley,0.632
19,Emerson Palmieri,Chelsea,emerson,0.609
42,William Fish,Manchester United,brandon williams,0.571


In [19]:
# ── Matches de score bajo confirmados manualmente ─────────────
ACCEPT_LOW_FUZZY = ['mahmoud trezeguet',
                    'bobby decordova reid',
                    'frank anguissa',
                    'wesley moraes',
                    'emerson palmieri',
                    'thiago alcantara'

]
# ─────────────────────────────────────────────────────────────

low_accepted = low_matches[low_matches['player_norm'].isin(ACCEPT_LOW_FUZZY)]
print(f'Aceptados del nivel bajo: {len(low_accepted)}')


Aceptados del nivel bajo: 6


### 7.4 Revisión muy estricta (score < 0.50)

Por defecto ninguno se acepta. Añadir a `ACCEPT_VERY_LOW_FUZZY` los correctos.

In [20]:
very_low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
45,Felipe Anderson,West Ham United,frederik alves,0.483
36,Allan Tchaptchet,Southampton,alex mccarthy,0.483
37,Kayne Ramsay,Southampton,alex mccarthy,0.480
23,Liam Delap,Manchester City,aymeric laporte,0.480
43,Nathan Broadhead,Everton,jarrad branthwaite,0.471
24,Niall Huggins,Leeds United,conor shaughnessy,0.467
29,Reda Khadra,Brighton & Hove Albion,alireza jahanbakhsh,0.467
44,Femi Seriki,Sheffield United,michael verrips,0.462
15,Antwoine Hackford,Sheffield United,oliver norwood,0.452
27,Joel Mumbongo,Burnley,josh brownhill,0.444


In [21]:
# ── Matches very low confirmados manualmente ──────────────────
ACCEPT_VERY_LOW_FUZZY = [

]
# ─────────────────────────────────────────────────────────────

very_low_accepted = very_low_matches[very_low_matches['player_norm'].isin(ACCEPT_VERY_LOW_FUZZY)]
print(f'Aceptados del nivel very low: {len(very_low_accepted)}')


Aceptados del nivel very low: 0


### 7.5 Aplicar todos los fuzzy matches aceptados

In [22]:
all_fuzzy    = pd.concat([auto_matches, review_accepted, low_accepted, very_low_accepted], ignore_index=True)
fuzzy_lookup = dict(zip(all_fuzzy['player_norm'], all_fuzzy['cg_match']))

df_merged['player_norm_fuzzy'] = df_merged.apply(
    lambda r: fuzzy_lookup.get(r['player_norm'], r['player_norm'])
    if pd.isna(r['gross_annual_eur']) else r['player_norm'],
    axis=1
)

df_final = (
    df_merged
    .drop(columns=['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality'])
    .merge(
        df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
               'position', 'age', 'nationality']],
        left_on=['player_norm_fuzzy', 'team_norm'],
        right_on=['player_norm', 'team_norm'],
        how='left'
    )
    .drop(columns=['player_norm_y', 'player_norm_fuzzy'])
    .rename(columns={'player_norm_x': 'player_norm'})
)

matched_final = df_final['gross_annual_eur'].notna().sum()
print(f'Resultado final: {matched_final}/{len(df_final)} ({matched_final/len(df_final):.1%})')
print(f'Sin salario:     {len(df_final) - matched_final}')

Resultado final: 494/527 (93.7%)
Sin salario:     33


## 8. Revisión de jugadores sin salario

Ordenados por equipo y minutos jugados para identificar si alguno debería tener salario.

In [23]:
sin_salario = (
    df_final[df_final['gross_annual_eur'].isna()]
    [['player', 'team', 'minutesPlayed', 'appearances', 'goals', 'assists']]
    .sort_values(['team', 'minutesPlayed'], ascending=[True, False])
    .reset_index(drop=True)
)

pd.set_option('display.max_rows', None)
print(f'Total sin salario: {len(sin_salario)}')
display(sin_salario)
pd.reset_option('display.max_rows')

Total sin salario: 33


,player,team,minutesPlayed,appearances,goals,assists
0,Carney Chukwuemeka,Aston Villa,25,2,0,0
1,Jaden Philogene-Bidace,Aston Villa,1,1,0,0
2,Reda Khadra,Brighton & Hove Albion,8,1,0,0
3,Josh Benson,Burnley,272,6,0,0
4,Joel Mumbongo,Burnley,37,4,0,0
5,Lewis Richardson,Burnley,3,2,0,0
6,Kepa Arrizabalaga,Chelsea,585,7,0,0
7,Moise Kean,Everton,13,2,0,0
8,Nathan Broadhead,Everton,2,1,0,0
9,Fabio Carvalho,Fulham,255,4,1,0


### 8.1 Comparación manual por equipo

Para cada equipo con jugadores sin salario se muestra la plantilla completa de Capology
ordenada alfabéticamente por nombre normalizado, facilitando la detección visual de matches fallidos.

In [24]:
equipos_sin_salario = sin_salario['team'].unique()

for equipo in sorted(equipos_sin_salario):
    sf_jugadores = sin_salario[sin_salario['team'] == equipo][['player', 'minutesPlayed']].sort_values('player')

    equipo_norm  = normalize(equipo)
    cg_jugadores = (
        df_cg[df_cg['team_norm'] == equipo_norm][['player', 'player_norm']]
        .sort_values('player_norm')
        .reset_index(drop=True)
    )

    print(f'\n{"="*60}')
    print(f'  {equipo}  —  SF sin salario:')
    display(sf_jugadores.reset_index(drop=True))
    print(f'  CG plantilla completa:')
    display(cg_jugadores)


  Aston Villa  —  SF sin salario:


,player,minutesPlayed
0,Carney Chukwuemeka,25
1,Jaden Philogene-Bidace,1


  CG plantilla completa:


,player,player_norm
0,Ahmed Elmohamady,ahmed elmohamady
1,Anwar El Ghazi,anwar el ghazi
2,Bertrand Traoré,bertrand traore
3,Björn Engels,bjorn engels
4,Conor Hourihane,conor hourihane
5,Douglas Luiz,douglas luiz
6,Emiliano Martínez,emiliano martinez
7,Ezri Konsa,ezri konsa
8,Frédéric Guilbert,frederic guilbert
9,Henri Lansbury,henri lansbury



  Brighton & Hove Albion  —  SF sin salario:


...

,player,player_norm
0,Aaron Cresswell,aaron cresswell
1,Andrii Yarmolenko,andrii yarmolenko
2,Angelo Ogbonna,angelo ogbonna
3,Arthur Masuaku,arthur masuaku
4,Ben Johnson,ben johnson
5,Craig Dawson,craig dawson
6,Darren Randolph,darren randolph
7,David Martin,david martin
8,Declan Rice,declan rice
9,Fabián Balbuena,fabian balbuena



  Wolverhampton  —  SF sin salario:


,player,minutesPlayed
0,Patrick Cutrone,23
1,Rúben Vinagre,172
2,Theo Corbeanu,8


  CG plantilla completa:


,player,player_norm
0,Adama Traoré,adama traore
1,Conor Coady,conor coady
2,Daniel Podence,daniel podence
3,Fábio Silva,fabio silva
4,João Moutinho,joao moutinho
5,John Ruddy,john ruddy
6,Jonny Otto,jonny otto
7,Ki-Jana Hoever,ki jana hoever
8,Leander Dendoncker,leander dendoncker
9,Luke Matheson,luke matheson


In [27]:
# ── Matches manuales (nombres muy distintos o traspasos invernales) ──
# Formato: (player_norm_sf, team_norm_sf): (player_norm_cg, team_norm_cg)
MANUAL_MATCHES = {
    ('kepa arrizabalaga', 'chelsea'): ('kepa', 'chelsea'),
}
# ────────────────────────────────────────────────────────────────────
print(f'Matches manuales definidos: {len(MANUAL_MATCHES)}')


Matches manuales definidos: 1


In [28]:
# Aplicar matches manuales sobre los que siguen sin salario
for (p_sf, t_sf), (p_cg, t_cg) in MANUAL_MATCHES.items():
    mask = (df_final['player_norm'] == p_sf) & (df_final['team_norm'] == t_sf) & (df_final['gross_annual_eur'].isna())
    datos_cg = df_cg[(df_cg['player_norm'] == p_cg) & (df_cg['team_norm'] == t_cg)]
    if not datos_cg.empty and mask.any():
        for col in ['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality']:
            df_final.loc[mask, col] = datos_cg[col].values[0]
        print(f'✅ Match manual aplicado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')
    else:
        print(f'⚠️  No encontrado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')

matched_tras_manual = df_final['gross_annual_eur'].notna().sum()
print(f'\nTras matches manuales: {matched_tras_manual}/{len(df_final)} ({matched_tras_manual/len(df_final):.1%})')

✅ Match manual aplicado: kepa arrizabalaga (chelsea) → kepa (chelsea)

Tras matches manuales: 495/527 (93.9%)


In [29]:
pd.reset_option('display.max_rows')

## 9. Guardado

Una vez revisado todo, se eliminan las columnas auxiliares y se guarda en `data/master/`.

In [30]:
df_final = df_final.drop(columns=['player_norm', 'team_norm'])

nombre_salida = 'master_england_2021.csv'
df_final.to_csv(MASTER_DIR / nombre_salida, index=False)

print(f'✅ Guardado: {nombre_salida}')
print(f'   Jugadores totales:  {len(df_final)}')
print(f'   Con salario:        {df_final["gross_annual_eur"].notna().sum()}')
print(f'   Sin salario (NaN):  {df_final["gross_annual_eur"].isna().sum()}')
print(f'   Columnas:           {df_final.shape[1]}')

✅ Guardado: master_england_2021.csv
   Jugadores totales:  527
   Con salario:        495
   Sin salario (NaN):  32
   Columnas:           121
